# TABLE 1. Baseline Characteristics
## v10 — FibroScan 변수 및 Follow-up duration 추가
- [FIX 1] `fu_min_days` default 265 → 365
- [FIX 2] Binary 분모: `len(s)` → `s.notna().sum()`
- [FIX 3] `fatty_pos`: `icd_fatty | imaging_fatty`
- [ADD 5] FibroScan (LSM, CAP) + Liver function (AST, ALT) rows 추가; CAP는 missing 수 병기
- [ADD 6] Follow-up duration (years) rows 추가; days → years 자동 변환

In [1]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

###############################################################################
# 0) 파일 설정
###############################################################################
DATA_PATH = "/home/ubuntu/Fibroscan_CKD_2025.xlsx"
SHEET_NAME = None 
OUTPUT_PATH = "Table1_Baseline_Characteristics_v10.xlsx"

HOSPITAL_COL = "Hospital"   # 10 = Sinchon, 20 = Gangnam

# cohort definition
MIN_FU_DAYS_FOR_COHORT = 365

# Table 1 cohort 선택
USE_FINAL_MODELING_COHORT = False
EVENT_COL = "CKD"
TTE_COL = "TTO_CKD"
FIVE_YEARS_DAYS = 1825

SHOW_CONTINUOUS_N = False

# Fatty liver 정의 상수 (275 vs 338)
CAP_STEATOSIS_CUTOFF = 238  # dB/m, S1 이상


###############################################################################
# 1) 유틸 함수
###############################################################################
def to_numeric_safe(series):
    return pd.to_numeric(series, errors="coerce")

def fmt_median_iqr(series, digits=2, show_n=False):
    s = to_numeric_safe(series).dropna()
    if len(s) == 0:
        return ""
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    med = s.median()
    if show_n:
        return f"{med:.{digits}f} [{q1:.{digits}f}, {q3:.{digits}f}] (n={len(s)})"
    return f"{med:.{digits}f} [{q1:.{digits}f}, {q3:.{digits}f}]"

def fmt_yes_n_pct_whole_cohort(series):
    s = to_numeric_safe(series)
    total = s.notna().sum()  # [FIX 2] 분모를 관측치(non-missing)로 수정 (missing을 No로 계산하는 오류 방지)
    if total == 0:
        return ""
    yes = (s == 1).sum()
    return f"{yes} ({yes / total * 100:.1f}%)"

def fmt_level_n_pct_whole_cohort(series, level):
    s = to_numeric_safe(series)
    total = len(s)
    if total == 0:
        return ""
    n = (s == level).sum()
    return f"{n} ({n / total * 100:.1f}%)"

def fmt_category_from_string_whole_cohort(series, category_value):
    total = len(series)
    if total == 0:
        return ""
    n = (series == category_value).sum()
    return f"{n} ({n / total * 100:.1f}%)"

def exists(df, col):
    return col in df.columns

###############################################################################
# 2) Study flowchart / cohort exclusion
###############################################################################
def exclusion_flow_table_by_hospital(
    df_raw: pd.DataFrame,
    hospital_col: str = "Hospital",
    egfr_col: str = "Baseline_eGFR",
    esrd_col: str = "ESRD",
    kt_col: str = "KT",
    proteinuria_col: str = "Baseline_PU",
    malignancy_col: str = "Malignancy",
    fu_col: str = "Follow-up_duration",
    fu_min_days: int = 365,  # [FIX 1] 오타 수정: 265 → 365 (MIN_FU_DAYS_FOR_COHORT와 일치)
) -> pd.DataFrame:
    df = df_raw.copy()

    for c in [egfr_col, esrd_col, kt_col, proteinuria_col, malignancy_col, fu_col]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    def crit_baseline_ckd(d):
        return (d[egfr_col] < 60) if egfr_col in d.columns else pd.Series(False, index=d.index)
    def crit_esrd(d):
        return (d[esrd_col] == 1) if esrd_col in d.columns else pd.Series(False, index=d.index)
    def crit_kt(d):
        return (d[kt_col] == 1) if kt_col in d.columns else pd.Series(False, index=d.index)
    def crit_proteinuria(d):
        return (d[proteinuria_col] == 1) if proteinuria_col in d.columns else pd.Series(False, index=d.index)
    def crit_malignancy(d):
        return (d[malignancy_col] == 1) if malignancy_col in d.columns else pd.Series(False, index=d.index)
    def crit_fu_lt_1Y(d):
        return (d[fu_col] < fu_min_days) if fu_col in d.columns else pd.Series(False, index=d.index)

    criteria = [
        ("Excl_Baseline_CKD_eGFR<60", crit_baseline_ckd),
        ("Excl_ESRD", crit_esrd),
        ("Excl_Kidney_Transplantation", crit_kt),
        ("Excl_Proteinuria", crit_proteinuria),
        ("Excl_Malignancy", crit_malignancy),
        ("Excl_FU<1Y", crit_fu_lt_1Y),
    ]

    rows = []
    for hosp, g in df.groupby(hospital_col):
        start_n = len(g)
        include_mask = pd.Series(True, index=g.index)
        step_counts = {}

        for name, crit_fn in criteria:
            to_exclude_now = include_mask & crit_fn(g).fillna(False)
            step_counts[name] = int(to_exclude_now.sum())
            include_mask &= ~to_exclude_now

        final_n = int(include_mask.sum())
        rows.append({
            "Hospital": hosp,
            "Start_N": start_n,
            **step_counts,
            "Final_Included_N": final_n,
            "Total_Excluded": start_n - final_n,
        })

    table = pd.DataFrame(rows).sort_values("Hospital").reset_index(drop=True)

    include_mask = pd.Series(True, index=df.index)
    all_steps = {}
    for name, crit_fn in criteria:
        to_excl = include_mask & crit_fn(df).fillna(False)
        all_steps[name] = int(to_excl.sum())
        include_mask &= ~to_excl

    all_row = {
        "Hospital": "ALL",
        "Start_N": len(df),
        **all_steps,
        "Final_Included_N": int(include_mask.sum()),
        "Total_Excluded": len(df) - int(include_mask.sum()),
    }

    table = pd.concat([table, pd.DataFrame([all_row])], ignore_index=True)
    return table

def apply_cohort_exclusions(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ["Baseline_eGFR", "ESRD", "KT", "Baseline_PU", "Malignancy", "Follow-up_duration"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    print(f"[EXCLUDE] Start n={len(out):,}")
    mask = pd.Series(True, index=out.index)

    if "Baseline_eGFR" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["Baseline_eGFR"] < 60)
        print(f"[EXCLUDE] Baseline CKD (eGFR<60): removed {(n_before - mask.sum()):,}")

    if "ESRD" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["ESRD"] == 1)
        print(f"[EXCLUDE] ESRD==1: removed {(n_before - mask.sum()):,}")

    if "KT" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["KT"] == 1)
        print(f"[EXCLUDE] KT==1: removed {(n_before - mask.sum()):,}")

    if "Baseline_PU" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["Baseline_PU"] == 1)
        print(f"[EXCLUDE] Baseline proteinuria (Baseline_PU==1): removed {(n_before - mask.sum()):,}")

    if "Malignancy" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["Malignancy"] == 1)
        print(f"[EXCLUDE] Malignancy==1: removed {(n_before - mask.sum()):,}")

    if "Follow-up_duration" in out.columns:
        n_before = mask.sum()
        mask &= ~(out["Follow-up_duration"] < MIN_FU_DAYS_FOR_COHORT)
        print(f"[EXCLUDE] FU<365days: removed {(n_before - mask.sum()):,}")

    out = out.loc[mask].reset_index(drop=True)
    print(f"[EXCLUDE] End n={len(out):,}")
    return out

###############################################################################
# 3) Optional: final modeling cohort
###############################################################################
def build_5yr_outcome(
    df: pd.DataFrame,
    event_col: str,
    tte_col: str,
    fu_col: str = "Follow-up_duration",
    five_year_days: int = FIVE_YEARS_DAYS,
) -> pd.DataFrame:
    out = df.copy()
    for col in [event_col, tte_col, fu_col]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    print(f"[OUTCOME] Building y for {event_col} within 5y (<= {five_year_days} days)")
    n0 = len(out)

    has_fu = out[fu_col].notna()
    event_is_1 = (out[event_col] == 1)
    has_tte_if_event = (~event_is_1) | (out[tte_col].notna())
    cand = out.loc[has_fu & has_tte_if_event].copy()

    print(f"[OUTCOME] After drop missing FU or missing TTE(if event): {len(cand):,} (removed {n0 - len(cand):,})")

    y = pd.Series(np.nan, index=cand.index)
    y.loc[(cand[event_col] == 1) & (cand[tte_col] > 0) & (cand[tte_col] <= five_year_days)] = 1
    y.loc[(cand[event_col] == 0) & (cand[fu_col] >= five_year_days)] = 0
    y.loc[(cand[event_col] == 1) & (cand[tte_col] > five_year_days) & (cand[fu_col] >= five_year_days)] = 0

    before_uncertain = len(cand)
    cand = cand.loc[y.notna()].copy()
    cand["y"] = y.loc[cand.index].astype(int)

    print(f"[OUTCOME] Excluded uncertain outcomes: removed {before_uncertain - len(cand):,}")
    print(f"[OUTCOME] Final labeled n={len(cand):,}, event rate={cand['y'].mean():.3f}")

    return cand.reset_index(drop=True)

###############################################################################
# 4) Etiology 분류
# 우선순위: HBV > HCV > Alcohol-related > Fatty liver > Unclassified
# HCV는 분류는 하되 Table 1에는 표시하지 않음
# → HCV 환자가 Unclassified로 섞이는 것을 방지
# Fatty liver 정의 (신뢰도 순 fallback):
#   1순위: CAP >= 238 dB/m (FibroScan 객관적 수치)
#   2순위: NAFLD_ICD == 1 (의사 진단 코드)
#   3순위: FattyLiver_US 또는 FattyLiver_CT == 1 (영상 소견)
# 분류 로직: 전부 isna() 가드 사용 → 한 번 배정된 환자는 재분류 불가
###############################################################################
def add_etiology_category(df):
    df = df.copy()

    # --- HBV ---
    hbv = to_numeric_safe(df["Baseline_HBsAg"]) if "Baseline_HBsAg" in df.columns \
        else pd.Series(np.nan, index=df.index)

    # --- HCV (분류용, Table 1 미표시) ---
    hcv = to_numeric_safe(df["Baseline_Anti_HCV"]) if "Baseline_Anti_HCV" in df.columns \
        else pd.Series(np.nan, index=df.index)

    # --- Alcohol ---
    alcoholic = to_numeric_safe(df["Alcoholic_ICD"]) if "Alcoholic_ICD" in df.columns \
        else pd.Series(0, index=df.index)

    # --- Fatty liver: 3단계 fallback ---

    # 1순위: CAP >= 238 dB/m
    cap_series = to_numeric_safe(df["CAP"]) if "CAP" in df.columns \
        else pd.Series(np.nan, index=df.index)
    has_cap   = cap_series.notna()
    cap_fatty = (cap_series >= CAP_STEATOSIS_CUTOFF).fillna(False)

    # 2순위: NAFLD ICD 진단 코드
    icd_fatty = (to_numeric_safe(df["NAFLD_ICD"]).fillna(0) == 1) if "NAFLD_ICD" in df.columns \
        else pd.Series(False, index=df.index)

    # 3순위: US / CT 영상 소견
    imaging_components = []
    for c in ["FattyLiver_US", "FattyLiver_CT"]:
        if c in df.columns:
            imaging_components.append(to_numeric_safe(df[c]).fillna(0))
    imaging_fatty = (sum(imaging_components) > 0) if len(imaging_components) > 0 \
        else pd.Series(False, index=df.index)

    # 최종 fatty flag: CAP 있으면 CAP 기준, 없으면 ICD OR 영상 (proxy끼리는 OR 허용)
    # [FIX 3] 구버전: np.where(icd_fatty, icd_fatty, ...) → icd_fatty 중복 참조 제거
    fatty_pos = pd.Series(
        np.where(has_cap,
                 cap_fatty,
                 icd_fatty | imaging_fatty),
        index=df.index,
        dtype=bool
    )

    # --- 조건 변수 정의 ---
    hbv_pos = (hbv == 1)
    hcv_pos = (hcv == 1)
    alc_pos = (alcoholic == 1)

    # --- 순차 배정: 전부 isna() 가드 적용 ---
    etiology = pd.Series(np.nan, index=df.index, dtype="object")

    etiology[etiology.isna() & hbv_pos]   = "HBV"
    etiology[etiology.isna() & hcv_pos]   = "HCV"          # 분류만, Table 1 미표시
    etiology[etiology.isna() & alc_pos]   = "Alcohol-related liver disease"
    etiology[etiology.isna() & fatty_pos] = "Fatty liver"
    etiology[etiology.isna()]             = "Unclassified / no documented chronic liver disease"

    df["Etiology"] = etiology

    # --- 디버그 출력 ---
    no_cap = ~has_cap
    print("[ETIOLOGY] 분포:")
    print(df["Etiology"].value_counts(dropna=False))
    print(f"\n  [Fatty liver 기준별 적용 현황]")
    print(f"  CAP 있는 환자:               {has_cap.sum():,} / {len(df):,}")
    print(f"  └ CAP 기준 fatty (>=238):    {cap_fatty[has_cap].sum():,}")
    print(f"  ICD 있는 환자 (CAP 없는 중): {icd_fatty[no_cap].sum():,}")
    print(f"  └ ICD 기준 fatty:            {icd_fatty[no_cap].sum():,}")
    print(f"  Imaging 기준 fatty (나머지): {imaging_fatty[no_cap & ~icd_fatty].sum():,}")

    return df

###############################################################################
# 5) 병원 이름
###############################################################################
def hospital_name(x):
    try:
        x = int(float(x))
    except Exception:
        return str(x)
    if x == 10:
        return "Sinchon Severance"
    elif x == 20:
        return "Gangnam Severance"
    else:
        return f"Hospital {x}"

###############################################################################
# 6) Baseline Table 1
###############################################################################
def make_table1_baseline(df):
    df = df.copy()
    df[HOSPITAL_COL] = pd.to_numeric(df[HOSPITAL_COL], errors="coerce")
    df = add_etiology_category(df)

    hospitals = [h for h in sorted(df[HOSPITAL_COL].dropna().unique())]
    hospital_labels = [hospital_name(h) for h in hospitals]

    subsets = {}
    counts = {}
    for h in hospitals:
        label = hospital_name(h)
        sub = df[df[HOSPITAL_COL] == h].copy()
        subsets[label] = sub
        counts[label] = len(sub)

    rows = [
        ("Demographics", None),
        ("Age (years)", ("cont", "Age")),
        ("Sex - Male", ("sex", 1)),
        ("Sex - Female", ("sex", 2)),
        ("BMI (kg/m²)", ("cont", "BMI")),

        ("Follow-up duration (years)", ("cont_years", "Follow-up_duration")),  # [ADD 6]

        ("Vital signs", None),
        ("SBP (mmHg)", ("cont", "SBP")),
        ("DBP (mmHg)", ("cont", "DBP")),
        ("Waist circumference (cm)", ("cont", "Waist")),

        ("Kidney function", None),
        ("Baseline eGFR (mL/min/1.73m²)", ("cont", "Baseline_eGFR")),
        ("Baseline creatinine (mg/dL)", ("cont", "Baseline_Cr")),

        ("Comorbidities", None),
        ("Diabetes mellitus (Yes)", ("bin", "DM")),
        ("Hypertension (Yes)", ("bin", "HTN")),

        ("FibroScan", None),                                                    # [ADD 5]
        ("LSM (kPa)", ("cont", "LSM_kPa")),
        ("CAP (dB/m)", ("cont_missing", "CAP")),

        ("Liver function", None),
        ("AST (IU/L)", ("cont", "Baseline_AST")),
        ("ALT (IU/L)", ("cont", "Baseline_ALT")),

        # HCV는 etiology 분류에는 포함되나 Table 1에는 미표시
        # Unclassified에 HCV 환자가 섞이지 않도록 처리됨
        ("Etiology", None),
        ("HBV", ("etiology", "HBV")),
        ("Alcohol-related liver disease", ("etiology", "Alcohol-related liver disease")),
        ("Fatty liver", ("etiology", "Fatty liver")),
        ("Unclassified / no documented chronic liver disease", ("etiology", "Unclassified / no documented chronic liver disease")),

        ("Metabolic / Lipid profile", None),
        ("Fasting plasma glucose (mg/dL)", ("cont", "Baseline_FPG")),
        ("HbA1c (%)", ("cont", "Baseline_HbA1c")),
        ("Triglycerides (mg/dL)", ("cont", "Baseline_TG")),
        ("Total cholesterol", ("cont", "Baseline_Cholesterol")),
        ("LDL cholesterol", ("cont", "Baseline_LDL")),
        ("HDL cholesterol", ("cont", "Baseline_HDL")),
        ("Uric acid (mg/dL)", ("cont", "Baseline_uricacid")),

        ("Medication", None),
        ("Anti-hypertensive medication (Yes)", ("bin", "BP_Med")),
        ("Anti-diabetic medication (Yes)", ("bin", "DM_Med")),
        ("SGLT2 inhibitor (Yes)", ("bin", "SGLT2i")),
        ("Statin (Yes)", ("bin", "Statin")),
    ]

    filtered_rows = []
    for row_label, spec in rows:
        if spec is None:
            filtered_rows.append((row_label, spec))
            continue

        kind, key = spec
        if kind in ("cont", "cont_years", "cont_missing") and exists(df, key):
            filtered_rows.append((row_label, spec))
        elif kind == "bin" and exists(df, key):
            filtered_rows.append((row_label, spec))
        elif kind == "sex" and exists(df, "Sex"):
            filtered_rows.append((row_label, spec))
        elif kind == "etiology" and exists(df, "Etiology"):
            filtered_rows.append((row_label, spec))

    final_rows = []
    i = 0
    while i < len(filtered_rows):
        row_label, spec = filtered_rows[i]
        if spec is None:
            if i + 1 < len(filtered_rows) and filtered_rows[i + 1][1] is not None:
                final_rows.append((row_label, spec))
        else:
            final_rows.append((row_label, spec))
        i += 1

    columns = ["Variable"] + hospital_labels
    out = pd.DataFrame("", index=range(len(final_rows)), columns=columns)

    for i, (row_label, spec) in enumerate(final_rows):
        out.loc[i, "Variable"] = row_label

        if spec is None:
            continue

        kind, key = spec
        for hlabel in hospital_labels:
            sub = subsets[hlabel]

            if kind == "cont":
                out.loc[i, hlabel] = fmt_median_iqr(sub[key], show_n=SHOW_CONTINUOUS_N)
            elif kind == "cont_years":   # [ADD 6] days → years 변환
                s_years = pd.to_numeric(sub[key], errors="coerce") / 365.25
                out.loc[i, hlabel] = fmt_median_iqr(s_years, digits=1, show_n=SHOW_CONTINUOUS_N)
            elif kind == "cont_missing": # [ADD 5] median [IQR] + missing n 표시
                s = pd.to_numeric(sub[key], errors="coerce")
                n_miss = s.isna().sum()
                base = fmt_median_iqr(s, show_n=SHOW_CONTINUOUS_N)
                out.loc[i, hlabel] = f"{base} (missing={n_miss})" if n_miss > 0 else base
            elif kind == "bin":
                out.loc[i, hlabel] = fmt_yes_n_pct_whole_cohort(sub[key])
            elif kind == "sex":
                out.loc[i, hlabel] = fmt_level_n_pct_whole_cohort(sub["Sex"], key)
            elif kind == "etiology":
                out.loc[i, hlabel] = fmt_category_from_string_whole_cohort(sub["Etiology"], key)

    n_row = pd.DataFrame([[""] + [f"N={counts[h]:,}" for h in hospital_labels]], columns=columns)
    out = pd.concat([n_row, out], ignore_index=True)

    return out, final_rows

###############################################################################
# 7) 엑셀 저장
###############################################################################
def save_table1_to_excel(table_df, final_rows, output_path):
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        table_df.to_excel(writer, sheet_name="Table1", index=False)

    wb = load_workbook(output_path)
    ws = wb["Table1"]

    navy_fill = PatternFill("solid", fgColor="203864")
    light_fill = PatternFill("solid", fgColor="E2EFDA")
    gray_fill  = PatternFill("solid", fgColor="D9E1F2")
    thin = Side(style="thin", color="D9D9D9")

    section_labels = {row_label for row_label, spec in final_rows if spec is None}

    for cell in ws[1]:
        cell.fill = navy_fill
        cell.font = Font(color="FFFFFF", bold=True, size=13)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for cell in ws[2]:
        cell.fill = gray_fill
        cell.font = Font(bold=False, size=12)
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for r in range(3, ws.max_row + 1):
        variable_value = ws.cell(r, 1).value
        is_section = variable_value in section_labels

        for c in range(1, ws.max_column + 1):
            cell = ws.cell(r, c)
            cell.border = Border(left=thin, right=thin, top=thin, bottom=thin)

            if is_section:
                cell.fill = light_fill
                cell.font = Font(bold=True, size=13)
                if c == 1:
                    cell.alignment = Alignment(horizontal="left", vertical="center")
                else:
                    cell.alignment = Alignment(horizontal="center", vertical="center")
            else:
                cell.font = Font(size=12)
                if c == 1:
                    cell.alignment = Alignment(horizontal="left", vertical="center", indent=1)
                else:
                    cell.alignment = Alignment(horizontal="center", vertical="center")

    ws.column_dimensions["A"].width = 48
    for c in range(2, ws.max_column + 1):
        ws.column_dimensions[get_column_letter(c)].width = 28

    for r in range(1, ws.max_row + 1):
        ws.row_dimensions[r].height = 24

    wb.save(output_path)

###############################################################################
# 8) 실행
###############################################################################
if __name__ == "__main__":
    if SHEET_NAME is None:
        df_raw = pd.read_excel(DATA_PATH)
    else:
        df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

    print(f"[RAW] n={len(df_raw):,}")

    flow_table = exclusion_flow_table_by_hospital(
        df_raw=df_raw,
        hospital_col=HOSPITAL_COL,
        egfr_col="Baseline_eGFR",
        esrd_col="ESRD",
        kt_col="KT",
        proteinuria_col="Baseline_PU",
        malignancy_col="Malignancy",
        fu_col="Follow-up_duration",
        fu_min_days=MIN_FU_DAYS_FOR_COHORT,
    )
    print("\n[FLOW TABLE]")
    print(flow_table)


    df_analysis = apply_cohort_exclusions(df_raw)
    print(f"[ANALYSIS COHORT] n={len(df_analysis):,}")

    if USE_FINAL_MODELING_COHORT:
        df_table = build_5yr_outcome(
            df=df_analysis,
            event_col=EVENT_COL,
            tte_col=TTE_COL,
            fu_col="Follow-up_duration",
            five_year_days=FIVE_YEARS_DAYS,
        )
        print(f"[FINAL MODELING COHORT for Table 1] n={len(df_table):,}")
    else:
        df_table = df_analysis.copy()
        print(f"[TABLE 1 COHORT = ANALYSIS COHORT] n={len(df_table):,}")

    table1, final_rows = make_table1_baseline(df_table)

    save_table1_to_excel(table1, final_rows, OUTPUT_PATH)
    print(f"Saved: {OUTPUT_PATH}")

[RAW] n=59,161

[FLOW TABLE]
  Hospital  Start_N  Excl_Baseline_CKD_eGFR<60  Excl_ESRD  \
0       10    44856                       1750         82   
1       20    14305                        389          5   
2      ALL    59161                       2139         87   

   Excl_Kidney_Transplantation  Excl_Proteinuria  Excl_Malignancy  Excl_FU<1Y  \
0                          120              4645             6335        3166   
1                            2              3568             1116        1197   
2                          122              8213             7451        4363   

   Final_Included_N  Total_Excluded  
0             28758           16098  
1              8028            6277  
2             36786           22375  
[EXCLUDE] Start n=59,161
[EXCLUDE] Baseline CKD (eGFR<60): removed 2,139
[EXCLUDE] ESRD==1: removed 87
[EXCLUDE] KT==1: removed 122
[EXCLUDE] Baseline proteinuria (Baseline_PU==1): removed 8,213
[EXCLUDE] Malignancy==1: removed 7,451
[EXCLUDE] FU<36

## Flow Diagram: Hospital-specific Exclusion Summary
Flow Diagram 슬라이드용 병원별 exclusion count 출력
- `flow_table`은 위 셀에서 이미 생성됨 (`exclusion_flow_table_by_hospital` 반환값)
- 이 셀만 단독 실행 시: 위 메인 셀을 먼저 실행해 `flow_table` 변수를 생성할 것


In [2]:
###############################################################################
# FLOW DIAGRAM — Hospital-specific Exclusion Summary
# flow_table: exclusion_flow_table_by_hospital() 반환값
# Hospital 10 = Sinchon Severance  |  Hospital 20 = Gangnam Severance
###############################################################################

def print_flow_diagram_summary(flow_table: "pd.DataFrame") -> None:
    """
    flow_table 의 Hospital=10, 20, ALL 행을 읽어
    Flow Diagram 슬라이드에 바로 쓸 수 있는 포맷으로 출력한다.
    """
    col_map = {
        "Excl_Baseline_CKD_eGFR<60":   "Baseline CKD (eGFR < 60)",
        "Excl_ESRD":                    "ESRD",
        "Excl_Kidney_Transplantation":  "Kidney transplantation",
        "Excl_Proteinuria":             "Proteinuria",
        "Excl_Malignancy":              "Malignancy",
        "Excl_FU<1Y":                   "Follow-up < 1 year",
    }

    hosp_display = {10: "Hos 10 (Sinchon Severance)",
                    20: "Hos 20 (Gangnam Severance)",
                    "ALL": "ALL (Combined)"}

    sep = "═" * 62
    thin = "─" * 62

    # ── 전체 raw N ──
    all_row  = flow_table[flow_table["Hospital"] == "ALL"].iloc[0]
    hos10    = flow_table[flow_table["Hospital"] == 10].iloc[0]
    hos20    = flow_table[flow_table["Hospital"] == 20].iloc[0]

    print(sep)
    print(f"  FLOW DIAGRAM: Study Population Selection")
    print(sep)
    print(f"  All patients undergoing FibroScan")
    print(f"  N = {int(all_row['Start_N']):,}  "
          f"(Sinchon: {int(hos10['Start_N']):,} | "
          f"Gangnam: {int(hos20['Start_N']):,})")
    print(thin)

    for excl_col, label in col_map.items():
        if excl_col not in flow_table.columns:
            continue
        n10  = int(hos10[excl_col])
        n20  = int(hos20[excl_col])
        nall = int(all_row[excl_col])
        print(f"  Excluded — {label}")
        print(f"    Hos 10 (Sinchon) : n = {n10:,}")
        print(f"    Hos 20 (Gangnam) : n = {n20:,}")
        print(f"    ALL              : n = {nall:,}")
        print()

    print(thin)
    print(f"  ▶ Final included cohort")
    print(f"    Hos 10 (Sinchon) : N = {int(hos10['Final_Included_N']):,}")
    print(f"    Hos 20 (Gangnam) : N = {int(hos20['Final_Included_N']):,}")
    print(f"    ALL              : N = {int(all_row['Final_Included_N']):,}")
    print(sep)


def flow_diagram_as_dataframe(flow_table: "pd.DataFrame") -> "pd.DataFrame":
    """
    같은 정보를 DataFrame 형태로 반환 (copy-paste / Excel 저장용).
    Columns: Criterion | Hos10_Sinchon | Hos20_Gangnam | ALL
    """
    col_map = {
        "Start_N":                       "Initial N (all FibroScan)",
        "Excl_Baseline_CKD_eGFR<60":    "Excluded: Baseline CKD (eGFR < 60)",
        "Excl_ESRD":                     "Excluded: ESRD",
        "Excl_Kidney_Transplantation":   "Excluded: Kidney transplantation",
        "Excl_Proteinuria":              "Excluded: Proteinuria",
        "Excl_Malignancy":               "Excluded: Malignancy",
        "Excl_FU<1Y":                    "Excluded: Follow-up < 1 year",
        "Final_Included_N":              "Final included N",
        "Total_Excluded":                "Total excluded",
    }

    hos10 = flow_table[flow_table["Hospital"] == 10].iloc[0]
    hos20 = flow_table[flow_table["Hospital"] == 20].iloc[0]
    all_r = flow_table[flow_table["Hospital"] == "ALL"].iloc[0]

    rows = []
    for raw_col, label in col_map.items():
        if raw_col not in flow_table.columns:
            continue
        rows.append({
            "Criterion": label,
            "Hos10_Sinchon": int(hos10[raw_col]),
            "Hos20_Gangnam": int(hos20[raw_col]),
            "ALL": int(all_r[raw_col]),
        })

    return pd.DataFrame(rows)


# ── 실행 ──
print_flow_diagram_summary(flow_table)
print()

flow_df = flow_diagram_as_dataframe(flow_table)
print("[DataFrame 형태]")
display(flow_df)

# 원하면 Excel로도 저장:
# flow_df.to_excel("FlowDiagram_ExclusionCounts.xlsx", index=False)
# print("Saved: FlowDiagram_ExclusionCounts.xlsx")


══════════════════════════════════════════════════════════════
  FLOW DIAGRAM: Study Population Selection
══════════════════════════════════════════════════════════════
  All patients undergoing FibroScan
  N = 59,161  (Sinchon: 44,856 | Gangnam: 14,305)
──────────────────────────────────────────────────────────────
  Excluded — Baseline CKD (eGFR < 60)
    Hos 10 (Sinchon) : n = 1,750
    Hos 20 (Gangnam) : n = 389
    ALL              : n = 2,139

  Excluded — ESRD
    Hos 10 (Sinchon) : n = 82
    Hos 20 (Gangnam) : n = 5
    ALL              : n = 87

  Excluded — Kidney transplantation
    Hos 10 (Sinchon) : n = 120
    Hos 20 (Gangnam) : n = 2
    ALL              : n = 122

  Excluded — Proteinuria
    Hos 10 (Sinchon) : n = 4,645
    Hos 20 (Gangnam) : n = 3,568
    ALL              : n = 8,213

  Excluded — Malignancy
    Hos 10 (Sinchon) : n = 6,335
    Hos 20 (Gangnam) : n = 1,116
    ALL              : n = 7,451

  Excluded — Follow-up < 1 year
    Hos 10 (Sinchon) : n = 3,

,Criterion,Hos10_Sinchon,Hos20_Gangnam,ALL
0,Initial N (all FibroScan),44856,14305,59161
1,Excluded: Baseline CKD (eGFR < 60),1750,389,2139
2,Excluded: ESRD,82,5,87
3,Excluded: Kidney transplantation,120,2,122
4,Excluded: Proteinuria,4645,3568,8213
5,Excluded: Malignancy,6335,1116,7451
6,Excluded: Follow-up < 1 year,3166,1197,4363
7,Final included N,28758,8028,36786
8,Total excluded,16098,6277,22375
